# Media-tower LoRA — change, control, round trip, selectivity, and the corrupt-row contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/24-media-tower-lora/media-tower-lora.ipynb)

Built from [`cookbook/book/chapters/24-media-tower-lora/media-tower-lora.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/24-media-tower-lora/media-tower-lora.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. On that GPU the
# chapter runs at `full` scale, over the published data; set SCALE = "small" to
# run the seconds-long version over the committed fixtures instead.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "full" if gpu else "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `fine_tune(method="lora", task=…, target_modules=…)` on an image, a text
and an audio tower · `encode_query` · `infer` · **Theory:** LoRA — a frozen base
weight plus a trainable low-rank update (Hu et al. 2022) — injected into OpenCLIP's
shared vision/text `ResidualAttentionBlock` (Radford et al. 2021) and into CLAP's
HTSAT-Swin audio tower (Wu et al. 2023) · **Rail:** measurement (five properties of a
tower adapter, each measured live).

Every other fine-tuning chapter trains a text encoder. This one adapts the media
towers with the same `fine_tune` verb: the vision and text towers of one OpenCLIP
checkpoint (`task=` alone chooses which) and the audio tower of a CLAP checkpoint,
each with `target_modules` naming real sites on that tower's own architecture —
`in_proj`/`c_fc` on OpenCLIP's attention block, `query`/`value`/`linear1` on
HTSAT-Swin. At `small` scale the towers are the committed random-weight fixtures;
at `full`, the real checkpoints. Either way the chapter measures five things,
because a single "the vector changed" check leaves four questions open:

1. **Change** — the adapter moves the served embedding. Direction and quality are
   not claimed: the corpora are synthetic shapes and tones with no semantic ground
   truth.
2. **Control** — the same input through the same checkpoint twice is identical, so
   the change is a parameter delta, not inference noise.
3. **Round trip** — the adapter survives the engine closing and a fresh engine
   opening over the same catalog: it is served from what was saved, not from a
   process's warm cache.
4. **Selectivity** — adapting one OpenCLIP tower leaves the other's served
   embedding untouched.
5. **The corrupt-row contract** — in a batch, a null and an undecodable row fail
   on their own row, and the good row is still served, whether the media column
   holds bytes or file paths.

In [ ]:
import tempfile
from pathlib import Path

import jammi
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from jammi_cookbook import contracts, encoders, fixtures, scale

SCALE = scale.current()
IMAGE_MODEL = encoders.image(SCALE)
AUDIO_MODEL = encoders.audio(SCALE)
work = Path(tempfile.mkdtemp())
home = f"file://{tempfile.mkdtemp()}"
db = jammi.connect(home)

images = {p.stem: p.read_bytes() for p in sorted(fixtures.path("tiny_image_corpus").glob("img_*.png"))}
clips = {p.stem: p.read_bytes() for p in sorted(fixtures.path("tiny_audio_corpus").glob("clip_*.wav"))}
print(f"{len(images)} images, {len(clips)} audio clips")

## Three towers, three adapters

A triplet source supervises each tower: an anchor, a positive of the same kind (the
next image of the same shape, the next clip of the same waveform) and a negative of
another kind. For the text tower the triplets are short captions of the same shapes.

In [ ]:
def kind(name: str) -> str:
    return name.split("_")[1]


def triplets(items: dict) -> dict:
    names = sorted(items)
    rows = {"anchor": [], "positive": [], "negative": []}
    for name in names:
        same = [n for n in names if kind(n) == kind(name) and n != name]
        other = next(n for n in names if kind(n) != kind(name))
        rows["anchor"].append(items[name])
        rows["positive"].append(items[same[0]])
        rows["negative"].append(items[other])
    return rows


captions = {f"cap_{kind(n)}_{i}": f"a {kind(n)} drawn in outline, view {i}"
            for i, n in enumerate(sorted(images))}
for name, rows in [("image_triplets", triplets(images)), ("text_triplets", triplets(captions)),
                   ("audio_triplets", triplets(clips))]:
    pq.write_table(pa.table(rows), work / f"{name}.parquet")
    db.add_source(name, url=str(work / f"{name}.parquet"), format="parquet")

TOWERS = {
    "vision": ("image_triplets", IMAGE_MODEL, "image_embedding", ["in_proj", "c_fc"]),
    "text": ("text_triplets", IMAGE_MODEL, "text_embedding", ["in_proj", "c_fc"]),
    "audio": ("audio_triplets", AUDIO_MODEL, "audio_embedding", ["query", "value", "linear1"]),
}


def adapt(source, base, task, sites) -> str:
    job = db.fine_tune(
        source=source, base_model=base, columns=["anchor", "positive", "negative"],
        method="lora", task=task, target_modules=sites, lora_rank=4, learning_rate=5e-3,
        epochs=2, batch_size=4, validation_fraction=0.0, early_stopping_metric="train_loss",
        seed=0,
    )
    job.wait()
    return job.output_model_id


tuned = {tower: adapt(*spec) for tower, spec in TOWERS.items()}
for tower, model in tuned.items():
    print(f"{tower:<7} -> {model}  ({db.describe_model(model)['task']})")

## 1 and 2 — the change, against the control

One probe per tower — an image, a caption, a clip — encoded through the base
checkpoint twice and through the adapted model once.

In [ ]:
PROBES = {
    "vision": ("image", images["img_circle_0"]),
    "text": ("text", "a circle drawn in outline"),
    "audio": ("audio", clips["clip_sine_0"]),
}


def encode(model: str, tower: str) -> np.ndarray:
    modality, probe = PROBES[tower]
    return np.asarray(db.encode_query(model=model, query=probe, modality=modality))


base = {tower: encode(spec[1], tower) for tower, spec in TOWERS.items()}
change, control = {}, {}
for tower, spec in TOWERS.items():
    control[tower] = float(np.abs(encode(spec[1], tower) - base[tower]).max())
    change[tower] = float(np.abs(encode(tuned[tower], tower) - base[tower]).max())
    print(f"{tower:<7} change max|Δ| {change[tower]:.6f}   control max|Δ| {control[tower]:.1f}")

In [ ]:
for tower in TOWERS:
    assert control[tower] == 0.0, (tower, control[tower])
    assert change[tower] > 0.0, tower
    contracts.assert_close(f"media_tower.{tower}.change_max_abs_diff", change[tower], tol=1e-4)

The control is exactly zero: the same checkpoint serves the same input
bit-identically, so every coordinate the adapter moved, it moved.

## 3 — the round trip through a fresh engine

The engine is closed — its model cache and every loaded adapter with it — and a
new one opens over the same catalog. The adapted models are resolved from the
catalog and the adapters reloaded from their saved bundles.

In [ ]:
served_before = {tower: encode(tuned[tower], tower) for tower in TOWERS}
db.close()
db = jammi.connect(home)
round_trip = {tower: float(np.abs(encode(tuned[tower], tower) - served_before[tower]).max())
              for tower in TOWERS}
for tower, diff in round_trip.items():
    print(f"{tower:<7} served before vs after the reopen: max|Δ| {diff:.1f}")

In [ ]:
assert all(diff == 0.0 for diff in round_trip.values()), round_trip

## 4 — adapting one tower leaves the other alone

The vision and text towers share one checkpoint and one site vocabulary; `task=`
scopes an adapter to one of them. The vision-adapted model's **text** embedding of
the caption, and the text-adapted model's **image** embedding of the image, are each
compared with the base checkpoint's.

In [ ]:
selectivity = {
    "vision adapter, text probe": float(np.abs(encode(tuned["vision"], "text") - base["text"]).max()),
    "text adapter, image probe": float(np.abs(encode(tuned["text"], "vision") - base["vision"]).max()),
}
for label, diff in selectivity.items():
    print(f"{label:<28} max|Δ| vs base {diff:.1f}")

In [ ]:
assert all(diff == 0.0 for diff in selectivity.values()), selectivity

Bit-identical, not merely close: the untouched tower is served exactly as the base
checkpoint serves it. This measures the outcome only — whether because training
touched only the tuned tower's sites or because serving applies an adapter only
under its own task, the served embedding of the other tower does not move.

## 5 — the corrupt-row contract, in both input arms

A media column arrives either as the encoded bytes or as a path to a file. Each arm
gets a batch of three rows: a valid one, a null, and one whose bytes decode as
nothing (an existing file of junk). `infer` returns one row per input with a
`_status`, and a bad row fails on its own row.

In [ ]:
junk = work / "junk.bin"
junk.write_bytes(b"not an image and not an audio clip")
good_image, good_clip = work / "good.png", work / "good.wav"
good_image.write_bytes(images["img_circle_0"])
good_clip.write_bytes(clips["clip_sine_0"])

ARMS = {
    ("image", "bytes"): pa.array([images["img_circle_0"], None, junk.read_bytes()], pa.binary()),
    ("image", "path"): pa.array([str(good_image), None, str(junk)], pa.string()),
    ("audio", "bytes"): pa.array([clips["clip_sine_0"], None, junk.read_bytes()], pa.binary()),
    ("audio", "path"): pa.array([str(good_clip), None, str(junk)], pa.string()),
}
contract = {}
for (modality, arm), column in ARMS.items():
    name = f"{modality}_{arm}"
    pq.write_table(pa.table({"id": [0, 1, 2], "media": column}), work / f"{name}.parquet")
    db.add_source(name, url=str(work / f"{name}.parquet"), format="parquet")
    model = IMAGE_MODEL if modality == "image" else AUDIO_MODEL
    out = db.infer(source=name, model=model, columns=["media"], task=f"{modality}_embedding",
                   key="id").sort_by("_row_id")
    contract[name] = list(zip(out.column("_status").to_pylist(), out.column("_error").to_pylist()))
    print(f"{name:<12} " + "  ".join(f"row {i}: {s}" for i, (s, _) in enumerate(contract[name])))
db.close()

In [ ]:
for name, rows in contract.items():
    (good, _), (null, null_error), (corrupt, corrupt_error) = rows
    assert (good, null, corrupt) == ("ok", "error", "error"), (name, rows)
    assert "null" in null_error.lower(), (name, null_error)
    assert "row 2" in corrupt_error, (name, corrupt_error)

The valid row is served in every arm; the null row names itself as null; the
undecodable row names its own position. A missing path — a file that does not exist
— is a different failure: resolving paths to bytes happens before any row is decoded,
so it refuses the whole call rather than one row.

## Bridge note

> **A tower adapter earns its place by five properties, not one.** The change is its
> reason to exist; the control makes that change interpretable; the round trip makes
> "saved" mean what a fresh engine can serve; selectivity keeps an adapter on its own
> tower; and the per-row contract keeps one bad input from failing a batch. Each is
> measured here through the same public surface a caller uses, on random fixtures at
> `small` and on real OpenCLIP and CLAP checkpoints at `full`.

## References

- Hu, Edward J., Shen, Yelong, Wallis, Phillip, Allen-Zhu, Zeyuan, Li, Yuanzhi, Wang, Shean, Wang, Lu, Chen, Weizhu (2022) *LoRA: Low-Rank Adaptation of Large Language Models* International Conference on Learning Representations (ICLR) arXiv:2106.09685.
- Radford, Alec, Kim, Jong Wook, Hallacy, Chris, Ramesh, Aditya, Goh, Gabriel, Agarwal, Sandhini, Sastry, Girish, Askell, Amanda, Mishkin, Pamela, Clark, Jack, Krueger, Gretchen, Sutskever, Ilya (2021) *Learning Transferable Visual Models From Natural Language Supervision* International Conference on Machine Learning (ICML) arXiv:2103.00020.
- Wu, Yusong, Chen, Ke, Zhang, Tianyu, Hui, Yuchen, Berg-Kirkpatrick, Taylor, Dubnov, Shlomo (2023) *Large-Scale Contrastive Language-Audio Pretraining with Feature Fusion and Keyword-to-Caption Augmentation* IEEE International Conference on Acoustics, Speech and Signal Processing (ICASSP) arXiv:2211.06687.